<a href="https://colab.research.google.com/github/felixyustian/enterprise_ai_context_engine_fraud_risk_marketing./blob/main/enterprise_ai_context_engine_fraud_risk_marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalasi dependensi dan penyiapan Redis
!pip install -q google-generativeai pandas numpy scikit-learn redis

# Install dan jalankan Redis server di background Colab
!apt-get install -y redis-server > /dev/null 2>&1
!redis-server --daemonize yes
!sleep 2

import pandas as pd
import numpy as np
import redis
import time
import json
import google.generativeai as genai
from sklearn.ensemble import IsolationForest
from typing import Dict

# Koneksi Redis
cache = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
print("Redis Server Connected:", cache.ping())

# Setup LLM API
GOOGLE_API_KEY = "MASUKKAN_GEMINI_API_KEY_ANDA_DI_SINI"
genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
# Real-World Data Ingestion (Google Drive) & ML Training
import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.ensemble import IsolationForest

# 1. Mount Google Drive
print("[INFO] Meminta akses ke Google Drive...")
drive.mount('/content/drive')

# 2. Definisikan path dataset publik Anda di Google Drive
# Sesuaikan path ini dengan lokasi file CSV Anda di dalam Drive
DATASET_PATH = '/content/drive/MyDrive/Enterprise_AI_Portfolio/transaction_data.csv'

print(f"\n[INFO] Mencoba memuat dataset dari: {DATASET_PATH}")

try:
    # Memuat data dari Google Drive
    df = pd.read_csv(DATASET_PATH)
    print(f"[SUCCESS] Dataset berhasil dimuat! Jumlah baris: {len(df)}")

    # -------------------------------------------------------------------
    # FALLBACK / PREPROCESSING LOGIC
    # Pastikan dataset memiliki kolom yang dibutuhkan oleh Pipeline MCP kita.
    # Jika Anda menggunakan dataset publik (misal dari Kaggle), Anda mungkin
    # perlu melakukan renaming kolom di bawah ini agar sesuai.
    #
    # Contoh pemetaan jika nama kolom di CSV berbeda:
    # df = df.rename(columns={'Amount': 'transaction_amount_usd', 'RiskScore': 'credit_score'})
    # -------------------------------------------------------------------

    # Simulasi injeksi fitur jika dataset publik tidak memiliki fitur spesifik ini
    if 'device_velocity' not in df.columns:
        print("[WARN] Fitur 'device_velocity' tidak ditemukan, membuat fitur sintetis berdasarkan distribusi poisson...")
        df['device_velocity'] = np.random.poisson(lam=1, size=len(df))

    if 'campaign_id' not in df.columns:
        print("[WARN] Fitur 'campaign_id' tidak ditemukan, memetakan traffic secara acak...")
        df['campaign_id'] = np.random.choice(['CAMP_A_IG', 'CAMP_B_TIKTOK', 'ORGANIC'], len(df), p=[0.4, 0.4, 0.2])

    # 3. Fraud Analytics: Train Isolation Forest (Menggunakan Data Real)
    print("\n[INFO] Melatih model Fraud Detection (Isolation Forest) pada data Google Drive...")

    # Ganti 'transaction_amount_usd' dengan nama kolom nominal transaksi di dataset Anda
    features = ['transaction_amount_usd', 'device_velocity']

    # Handle missing values jika ada pada dataset publik
    df[features] = df[features].fillna(df[features].median())

    iso_forest = IsolationForest(contamination=0.05, random_state=42)
    df['is_fraud_suspected'] = iso_forest.fit_predict(df[features])
    df['is_fraud_suspected'] = df['is_fraud_suspected'].map({1: False, -1: True})

    # 4. Risk Analytics: Hitung Risk Profile
    # Sesuaikan threshold dengan distribusi 'credit_score' pada dataset Anda
    df['high_risk_credit'] = df['credit_score'] < 550

    print("\n[INFO] Data Pipeline Selesai. Sample Data:")
    display(df.head())

except FileNotFoundError:
    print(f"\n[ERROR] File tidak ditemukan di path: {DATASET_PATH}")
    print("-> Mohon pastikan Anda telah membuat folder 'Enterprise_AI_Portfolio' di Google Drive utama Anda.")
    print("-> Dan pastikan file dataset bernama 'transaction_data.csv' sudah di-upload ke folder tersebut.")
except Exception as e:
    print(f"\n[ERROR] Terjadi kesalahan saat memproses data: {e}")

In [ ]:
# Definisi Tooling / Model Context Protocol (MCP)

def get_marketing_roi(campaign_id: str) -> str:
    """
    Mengambil data efektivitas marketing dan total volume transaksi dari sebuah kampanye pemasaran.
    Gunakan ini untuk mengevaluasi apakah sebuah kampanye menghasilkan revenue.
    """
    camp_data = df[df['campaign_id'] == campaign_id]
    if camp_data.empty:
        return json.dumps({"error": "Campaign not found"})

    total_volume = camp_data['transaction_amount_usd'].sum()
    total_users = camp_data['user_id'].nunique()

    return json.dumps({
        "campaign_id": campaign_id,
        "total_revenue_usd": round(total_volume, 2),
        "unique_customers_acquired": total_users,
        "avg_transaction_value": round(total_volume / len(camp_data), 2)
    })

def get_fraud_and_risk_report(campaign_id: str) -> str:
    """
    Menganalisis indikasi penipuan (fraud) dan risiko kredit (credit risk) dari akuisisi pengguna pada kampanye tertentu.
    Gunakan ini untuk memvalidasi kualitas traffic dari sebuah kampanye.
    """
    camp_data = df[df['campaign_id'] == campaign_id]
    if camp_data.empty:
        return json.dumps({"error": "Campaign not found"})

    total_transactions = len(camp_data)
    fraud_cases = camp_data['is_fraud_suspected'].sum()
    high_risk_users = camp_data['high_risk_credit'].sum()

    return json.dumps({
        "campaign_id": campaign_id,
        "total_transactions": total_transactions,
        "fraudulent_transactions_detected": int(fraud_cases),
        "fraud_rate_percentage": round((fraud_cases / total_transactions) * 100, 2),
        "high_credit_risk_users": int(high_risk_users),
        "risk_rate_percentage": round((high_risk_users / total_transactions) * 100, 2)
    })

# Daftarkan tools ke model AI
mcp_tools = [get_marketing_roi, get_fraud_and_risk_report]
model = genai.GenerativeModel(
    model_name='gemini-2.5-pro',
    tools=mcp_tools
)

In [ ]:
# LLM Orchestration & Business Execution

def executive_ai_analyst(query: str) -> str:
    """Menjalankan kueri bisnis menggunakan LLM dan Redis Cache."""
    cache_key = f"exec_query:{query.replace(' ', '_').lower()[:50]}"

    if cached_result := cache.get(cache_key):
        print("⚡ [CACHE HIT] Mengambil dari Redis...\n")
        return cached_result

    print("🧠 [COMPUTING] LLM sedang menganalisis data lintas departemen...")
    chat = model.start_chat()

    prompt = (
        f"Anda adalah AI R&D Manager dan Lead Data Scientist. "
        f"Pertanyaan eksekutif: {query}\n\n"
        "Instruksi Eksekusi:\n"
        "1. Panggil tool marketing untuk melihat performa awal.\n"
        "2. WAJIB panggil tool fraud & risk untuk memvalidasi kualitas traffic kampanye tersebut.\n"
        "3. Berikan laporan komprehensif, temuan anomali, dan rekomendasi strategis (hentikan kampanye, audit, atau scale-up).\n"
        "Gunakan bahasa profesional."
    )

    response = chat.send_message(prompt)
    result = response.text

    # Simpan ke cache (TTL 1 Jam)
    cache.setex(cache_key, 3600, result)
    return result

# --- PENGUJIAN ---
business_query = "Tolong evaluasi performa kampanye 'CAMP_B_TIKTOK'. Apakah traffic yang masuk berkualitas, atau kita menghadapi risiko fraud dan kredit macet? Apa rekomendasi strategisnya?"

print("="*60)
print(f"💼 BUSINESS QUERY: {business_query}")
print("="*60)

final_strategy = executive_ai_analyst(business_query)

print("\n📊 EXECUTIVE SUMMARY & STRATEGY:\n")
print(final_strategy)